<a href="https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/codex/fix-madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QuickStart CityLearn v3 MADRL

Este notebook sigue la estructura del `quickstart.ipynb` original de CityLearn, pero esta adaptado al proyecto **CityLearn v3 MADRL**. El simulador base sigue siendo CityLearn v2, y la capa v3 agrega Dec-POMDP, CTDE, 17 edificios + EV, tres ejes `E1/E2/E3`, recompensa `CityLearnV3MADRLRewardFunction`, KPIs por eje y backends oficiales HAPPO, MASAC, MATD3 y MAAC.

En local se recomienda usar el entorno `.venv39-citylearn-v3`. En Google Colab se debe clonar la rama `citylearn-v3-madrl` e instalar el repositorio en modo editable. La instalacion queda apagada por defecto.

In [ ]:
# Controla si se instala el proyecto en Google Colab.
RUN_COLAB_INSTALL = False

# Importa utilidades del sistema operativo.
import os

# Importa rutas portables para Windows, Linux y Colab.
from pathlib import Path

# Ejecuta comandos externos cuando una celda lo requiere.
import subprocess

# Expone el interprete Python activo.
import sys

# Importa display cuando se ejecuta dentro de Jupyter.
try:
    # Usa el display nativo de IPython.
    from IPython.display import display
# Define un fallback para validaciones ejecutadas como script.
except Exception:
    # Usa print cuando IPython no esta disponible.
    display = print

# Instala el proyecto solo si se activa la bandera.
if RUN_COLAB_INSTALL:
    # Clona la rama publica del proyecto CityLearn v3 MADRL.
    subprocess.run(['git', 'clone', '--branch', 'citylearn-v3-madrl', 'https://github.com/Mac-Tapia/CityLearn.git'], check=True)
    # Cambia el directorio activo al repositorio clonado.
    os.chdir('CityLearn')
    # Instala CityLearn en modo editable.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
# Mantiene la celda segura cuando no se instala nada.
else:
    # Informa que no se modifico el entorno.
    print('RUN_COLAB_INSTALL=False; usando el entorno actual.')

## Project Setup

La siguiente celda detecta la raiz CityLearn aunque el notebook se ejecute desde `examples/`, desde `CityLearn/` o desde el superproyecto `MADRLCitytleranflexresdr/`.

In [ ]:
# Obtiene el directorio actual del notebook.
current_path = Path.cwd().resolve()

# Detecta la raiz CityLearn cuando se ejecuta desde examples.
if current_path.name == 'examples' and (current_path.parent / 'scripts').is_dir():
    # Usa la carpeta padre como raiz CityLearn.
    project_root = current_path.parent
# Detecta la raiz CityLearn cuando se ejecuta desde CityLearn.
elif (current_path / 'scripts').is_dir() and (current_path / 'citylearn').is_dir():
    # Usa el directorio actual como raiz CityLearn.
    project_root = current_path
# Detecta el superproyecto que contiene el subdirectorio CityLearn.
elif (current_path / 'CityLearn' / 'scripts').is_dir():
    # Usa el subdirectorio CityLearn como raiz ejecutable.
    project_root = current_path / 'CityLearn'
# Usa el directorio actual como ultimo recurso.
else:
    # Deja que las validaciones posteriores muestren cualquier problema de ruta.
    project_root = current_path

# Agrega la raiz CityLearn al path de Python si no esta instalada como paquete editable.
if str(project_root) not in sys.path:
    # Permite importar citylearn desde el codigo fuente local.
    sys.path.insert(0, str(project_root))

# Define la carpeta de scripts oficiales.
scripts_dir = project_root / 'scripts'

# Define la carpeta de salidas quickstart.
quickstart_output = project_root / 'outputs' / 'citylearn_v3_madrl_quickstart'

# Define el dataset oficial del proyecto.
dataset_name = 'citylearn_iquitos_2023_2025'

# Define la ruta local versionada del schema.
schema_path = project_root / 'data' / 'datasets' / dataset_name / 'schema.json'

# Falla temprano si el dataset actual no esta disponible.
if not schema_path.is_file():
    raise FileNotFoundError(f'No se encontro el schema del proyecto: {schema_path}')

# Define el horizonte corto del quickstart.
quick_steps = 24

# Define la semilla reproducible.
seed = 0

# Muestra la raiz detectada.
print('project_root =', project_root)

# Muestra la carpeta de scripts.
print('scripts_dir =', scripts_dir)

# Muestra la carpeta de salida.
print('quickstart_output =', quickstart_output)

# Muestra el schema activo.
print('schema_path =', schema_path)

## CityLearn Control Agents

El QuickStart original muestra agentes CityLearn internos como baseline, RBC, SAC y MARLISA. En este proyecto esos agentes se conservan como referencia v2, pero la contribucion principal es entrenar controladores MADRL v3 con Dec-POMDP y CTDE.

### No Control Baseline

Esta celda ejecuta el baseline CityLearn v2 en el mismo dataset del proyecto. Se usa `episode_time_steps=24` para que el QuickStart sea rapido; el entrenamiento oficial usa 8760 pasos.

In [ ]:
# Importa el agente baseline de CityLearn v2.
from citylearn.agents.base import BaselineAgent as BaselineAgent

# Importa el entorno base CityLearn v2.
from citylearn.citylearn import CityLearnEnv

# Crea el entorno centralizado de baseline.
baseline_env = CityLearnEnv(str(schema_path), central_agent=True, episode_time_steps=quick_steps, random_seed=seed, offline=True)

# Crea el agente baseline sin control activo.
baseline_model = BaselineAgent(baseline_env)

# Reinicia el entorno.
observations, _ = baseline_env.reset(seed=seed)

# Ejecuta un episodio corto.
while not baseline_env.terminated:
    # Predice acciones baseline.
    actions = baseline_model.predict(observations)
    # Aplica acciones al entorno.
    observations, reward, terminated, truncated, info = baseline_env.step(actions)

# Evalua KPIs CityLearn v2.
baseline_kpis = baseline_env.evaluate()

# Convierte KPIs a una tabla compacta.
baseline_table = baseline_kpis.pivot(index='cost_function', columns='name', values='value').round(3).dropna(how='all')

# Muestra la tabla baseline.
display(baseline_table)

### Centralized RBC

RBC se usa como linea base explicable. Para el dataset actual se usa el controlador de referencia con soporte EV y `washing_machine_1`, porque `BasicRBC` no cubre todos los actuadores de `citylearn_iquitos_2023_2025`.

In [ ]:
# Importa RBC de referencia con soporte EV y cargas flexibles.
from citylearn.agents.rbc import BasicElectricVehicleRBC_ReferenceController as ProjectRBC

# Crea el entorno centralizado para RBC.
rbc_env = CityLearnEnv(str(schema_path), central_agent=True, episode_time_steps=quick_steps, random_seed=seed, offline=True)

# Crea el controlador RBC compatible con el dataset actual.
rbc_model = ProjectRBC(rbc_env)

# Reinicia el entorno.
observations, _ = rbc_env.reset(seed=seed)

# Ejecuta un episodio corto.
while not rbc_env.terminated:
    # Predice acciones RBC.
    actions = rbc_model.predict(observations)
    # Aplica acciones al entorno.
    observations, reward, terminated, truncated, info = rbc_env.step(actions)

# Evalua KPIs CityLearn v2.
rbc_kpis = rbc_env.evaluate()

# Convierte KPIs a tabla compacta.
rbc_table = rbc_kpis.pivot(index='cost_function', columns='name', values='value').round(3).dropna(how='all')

# Muestra la tabla RBC.
display(rbc_table)

### CityLearn v3 Dec-POMDP Smoke Test

Esta celda crea el entorno v3 del proyecto. La diferencia clave frente a CityLearn v2 es que ahora cada edificio es un agente, el estado global se expone para CTDE y la recompensa usa pesos por eje y perfil MADRL.

In [ ]:
# Importa utilidades del entorno v3.
from citylearn.v3 import describe_environment, evaluate_objectives, make_citylearn_v3_project_env

# Crea el entorno Dec-POMDP para el eje E1 con perfil HAPPO.
madrl_env = make_citylearn_v3_project_env(scenario='E1', seed=seed, episode_time_steps=quick_steps, madrl_algorithm='HAPPO')

# Describe el entorno v3 creado.
madrl_description = describe_environment(madrl_env)

# Imprime numero de agentes.
print('num_agents:', madrl_description['num_agents'])

# Imprime dimension del estado global CTDE.
print('state_dim:', madrl_description['state_dim'])

# Imprime si hay acciones EV.
print('has_ev_actions:', madrl_description['has_ev_actions'])

# Imprime metadatos de reward v3.
print('reward_metadata:', madrl_description['reward_metadata'])

# Reinicia el entorno multiagente.
observations, infos = madrl_env.reset(seed=seed)

# Construye acciones cero validas por agente.
zero_actions = {agent: madrl_env.action_space(agent).sample() * 0.0 for agent in madrl_env.agents}

# Ejecuta un paso Dec-POMDP.
next_observations, rewards, terminations, truncations, step_infos = madrl_env.step(zero_actions)

# Imprime recompensa media del primer paso.
print('reward_mean_step_1:', sum(rewards.values()) / max(len(rewards), 1))

# Imprime forma del estado CTDE.
print('state_shape:', madrl_env.state().shape)

### Decentralized-Cooperative MADRL: HAPPO

HAPPO usa entrenamiento on-policy y critic centralizado desde HARL, con ejecucion descentralizada por edificio. La celda imprime un comando smoke; no ejecuta entrenamiento salvo que se active `RUN_HAPPO_SMOKE`.

In [ ]:
# Controla si se ejecuta una prueba smoke de HAPPO.
RUN_HAPPO_SMOKE = False

# Construye el comando HAPPO corto.
happo_command = [sys.executable, '-B', str(scripts_dir / 'train_citylearn_v3_happo.py'), '--scenario', 'E1', '--seed', str(seed), '--episode-time-steps', '4', '--episodes', '1', '--num-env-steps', '4', '--hidden-size', '256', '--torch-threads', '4', '--output-dir', str(quickstart_output / 'happo')]

# Muestra el comando HAPPO.
print(' '.join(map(str, happo_command)))

# Ejecuta HAPPO solo si se activa la bandera.
if RUN_HAPPO_SMOKE:
    # Lanza una prueba minima de HAPPO.
    subprocess.run(happo_command, check=True, cwd=project_root)
# Mantiene la celda segura por defecto.
else:
    # Informa que no se ejecuto HAPPO.
    print('RUN_HAPPO_SMOKE=False; comando mostrado sin ejecutar.')

### Decentralized-Cooperative MADRL: MASAC

MASAC usa una variante soft actor-critic multiagente con estado global CTDE y politica local discretizada para acoplarse a CityLearn.

In [ ]:
# Controla si se ejecuta una prueba smoke de MASAC.
RUN_MASAC_SMOKE = False

# Construye el comando MASAC corto.
masac_command = [sys.executable, '-B', str(scripts_dir / 'train_citylearn_v3_masac.py'), '--scenario', 'E2', '--seed', str(seed), '--episode-time-steps', '4', '--episodes', '1', '--action-bins', '3', '--buffer-size', '2', '--output-dir', str(quickstart_output / 'masac')]

# Muestra el comando MASAC.
print(' '.join(map(str, masac_command)))

# Ejecuta MASAC solo si se activa la bandera.
if RUN_MASAC_SMOKE:
    # Lanza una prueba minima de MASAC.
    subprocess.run(masac_command, check=True, cwd=project_root)
# Mantiene la celda segura por defecto.
else:
    # Informa que no se ejecuto MASAC.
    print('RUN_MASAC_SMOKE=False; comando mostrado sin ejecutar.')

### Decentralized-Cooperative MADRL: MATD3

MATD3 usa actores continuos y criticos centralizados con observaciones/acciones conjuntas. Es el backend mas natural para control continuo de almacenamiento, EV y rampas.

In [ ]:
# Controla si se ejecuta una prueba smoke de MATD3.
RUN_MATD3_SMOKE = False

# Construye el comando MATD3 corto.
matd3_command = [sys.executable, '-B', str(scripts_dir / 'train_citylearn_v3_matd3.py'), '--scenario', 'E3', '--seed', str(seed), '--episode-time-steps', '4', '--episodes', '1', '--num-env-steps', '4', '--batch-size', '4', '--buffer-size', '128', '--hidden-size', '64', '--train-interval', '1', '--num-random-episodes', '1', '--output-dir', str(quickstart_output / 'matd3')]

# Muestra el comando MATD3.
print(' '.join(map(str, matd3_command)))

# Ejecuta MATD3 solo si se activa la bandera.
if RUN_MATD3_SMOKE:
    # Lanza una prueba minima de MATD3.
    subprocess.run(matd3_command, check=True, cwd=project_root)
# Mantiene la celda segura por defecto.
else:
    # Informa que no se ejecuto MATD3.
    print('RUN_MATD3_SMOKE=False; comando mostrado sin ejecutar.')

### Decentralized-Cooperative MADRL: MAAC

MAAC usa critic de atencion para coordinar agentes. En este proyecto se adapta a la comunidad de edificios con acciones discretizadas y observaciones locales por edificio.

In [ ]:
# Controla si se ejecuta una prueba smoke de MAAC.
RUN_MAAC_SMOKE = False

# Construye el comando MAAC corto.
maac_command = [sys.executable, '-B', str(scripts_dir / 'train_citylearn_v3_maac.py'), '--scenario', 'E1', '--seed', str(seed), '--episode-time-steps', '4', '--episodes', '1', '--action-bins', '3', '--batch-size', '4', '--buffer-length', '256', '--steps-per-update', '1', '--num-updates', '1', '--hidden-size', '128', '--attend-heads', '4', '--output-dir', str(quickstart_output / 'maac')]

# Muestra el comando MAAC.
print(' '.join(map(str, maac_command)))

# Ejecuta MAAC solo si se activa la bandera.
if RUN_MAAC_SMOKE:
    # Lanza una prueba minima de MAAC.
    subprocess.run(maac_command, check=True, cwd=project_root)
# Mantiene la celda segura por defecto.
else:
    # Informa que no se ejecuto MAAC.
    print('RUN_MAAC_SMOKE=False; comando mostrado sin ejecutar.')

## Other Standard Reinforcement Learning Libraries

El QuickStart original incluye Stable-Baselines3 y RLlib. Para CityLearn v3 MADRL, los backends oficiales del proyecto viven en `external/` y los scripts `train_citylearn_v3_*.py`. Aun asi, el proyecto incluye adaptadores para flujos RLlib/MARLlib cuando se requieran experimentos adicionales.

### Stable Baselines3 Reinforcement Learning Algorithms

Stable-Baselines3 no soporta multiagente Dec-POMDP directamente. Se puede usar como baseline centralizado CityLearn v2, pero no representa la contribucion MADRL v3. Por eso esta celda solo imprime el comando de instalacion opcional.

In [ ]:
# Controla si se instala Stable-Baselines3.
INSTALL_SB3 = False

# Construye el comando de instalacion compatible con CityLearn v2.
sb3_install_command = [sys.executable, '-m', 'pip', 'install', 'stable-baselines3<=2.2.1']

# Muestra el comando opcional.
print(' '.join(sb3_install_command))

# Instala SB3 solo si se activa la bandera.
if INSTALL_SB3:
    # Ejecuta la instalacion opcional.
    subprocess.run(sb3_install_command, check=True)
# Mantiene la celda segura por defecto.
else:
    # Informa que no se instalo SB3.
    print('INSTALL_SB3=False; SB3 no es necesario para los cuatro MADRL oficiales.')

### RLlib / MARLlib Adapter

`CityLearnV3MARLlibEnv` expone el Dec-POMDP como `MultiAgentEnv`: rellena observaciones/acciones heterogeneas a dimensiones comunes y mantiene el mapeo por edificio. Esta celda prueba el adaptador sin entrenar RLlib.

In [ ]:
# Importa la configuracion v3 del proyecto.
from citylearn.v3 import CityLearnV3ExperimentConfig, CityLearnV3MARLlibEnv

# Crea una configuracion corta para el adaptador.
marllib_config = CityLearnV3ExperimentConfig(episode_time_steps=quick_steps)

# Crea el adaptador RLlib/MARLlib para E1.
marllib_env = CityLearnV3MARLlibEnv({'config': marllib_config, 'scenario': 'E1', 'seed': seed})

# Obtiene informacion multiagente del entorno.
env_info = marllib_env.get_env_info()

# Imprime numero de agentes.
print('num_agents:', env_info['num_agents'])

# Imprime limite de episodio.
print('episode_limit:', env_info['episode_limit'])

# Imprime dimensiones originales de los primeros agentes.
print('original_observation_dims_sample:', dict(list(env_info['original_observation_dims'].items())[:3]))

# Reinicia el adaptador para validar observaciones.
marllib_observations = marllib_env.reset(seed=seed)

# Imprime cantidad de observaciones multiagente.
print('observations:', len(marllib_observations))

## Official Full Training

El entrenamiento completo oficial no debe lanzarse accidentalmente desde un QuickStart. Esta celda imprime el comando PowerShell que ejecuta los tres ejes y los cuatro MADRL con hiperparametros oficiales del proyecto.

In [ ]:
# Define la salida oficial v4 (entrenamiento activo desde 2026-06-08).
official_output = project_root / 'outputs' / 'citylearn_v3_madrl_oficial_v4'

# Construye el comando oficial PowerShell con perfil local4060 y SkipCompleted.
official_command = [
    'powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass',
    '-File', str(scripts_dir / 'launch_citylearn_v3_official_training.ps1'),
    '-Scenario', 'ALL',
    '-Seed', str(seed),
    '-EpisodeTimeSteps', '8760',
    '-Episodes', '5',
    '-OutputRoot', str(official_output),
    '-GpuProfile', 'local4060',
    '-TorchThreads', '12',
    '-Cuda',
    '-LiveOutput',
    '-SkipCompleted',
]

# Muestra el comando oficial.
print(' '.join(map(str, official_command)))

## Neighborhood Dataset Generation

El QuickStart original muestra como generar barrios residenciales con EnergyPlus. Esa funcionalidad sigue perteneciendo a CityLearn v2. Para este proyecto, cualquier nuevo `schema.json` compatible con CityLearn v2 puede entrar a la capa v3 mediante `make_citylearn_v3_env(schema_path=...)` y conservar Dec-POMDP, CTDE, reward v3 y KPIs por eje.

In [ ]:
# Importa el schema oficial local del proyecto.
from citylearn.dec_pomdp import DEFAULT_17_BUILDING_EV_SCHEMA

# Importa la fabrica generica de CityLearn v3.
from citylearn.v3 import make_citylearn_v3_env

# Usa el schema actual como ejemplo de entrada generica.
custom_schema_path = DEFAULT_17_BUILDING_EV_SCHEMA

# Crea un entorno v3 desde un schema explicito.
custom_env = make_citylearn_v3_env(schema_path=custom_schema_path, scenario='E2', seed=seed, episode_time_steps=quick_steps, madrl_algorithm='MASAC')

# Resume el entorno construido desde schema explicito.
custom_description = describe_environment(custom_env)

# Imprime ruta del schema usado.
print('schema_path:', custom_schema_path)

# Imprime escenario activo.
print('scenario:', custom_description['scenario'])

# Imprime reward activo.
print('reward_function:', custom_description['reward_function'])

# Imprime perfil de reward.
print('reward_profile:', custom_description['reward_metadata']['profile']['profile_name'])

## Quick Evaluation by Project Axes

La evaluacion final del proyecto no se basa solo en reward. Se reportan KPIs CityLearn v2 agrupados por los tres ejes: flexibilidad energetica, emisiones de CO2 y costos energeticos.

In [ ]:
# Evalua objetivos sobre el entorno v3 creado para smoke test.
objective_report = evaluate_objectives(madrl_env)

# Recorre los tres ejes del proyecto.
for axis_name, axis_payload in objective_report['axes'].items():
    # Extrae el resumen de comparacion contra baseline.
    comparison = axis_payload['baseline_comparison']
    # Imprime resumen por eje.
    print(axis_name, comparison)